In [7]:
import sys
import os

%load_ext autoreload
%autoreload 2

repo_dir = os.path.dirname(os.path.abspath("./"))
if repo_dir not in sys.path:
    print(f'add repository dir: {repo_dir}')
    sys.path.append(repo_dir)


from rl.retrieval_babilong import RetrNoiseInjectionDataset, RetrSentenceSampler
from babilong_utils import TaskDataset

import datasets
from datasets import Dataset, load_dataset, load_from_disk
import torch
import sys
import time
import numpy as np
from collections import deque

# os.environ["CUDA_LAUNCH_BLOCKING"] = "1"


max_steps = 6
num_sentences = 50
task = "qa2_two-supporting-facts"
noise_train_path = "/home/nazar/pg19-train-with-sentences"
noise_path_test = "/home/nazar/pg19-test-with-sentences"
facts_train_path = f"/home/nazar/tasks_1-20_v1-2/en-10k/{task}_train.txt"
facts_test_path = f"/home/nazar/tasks_1-20_v1-2/en-10k/{task}_test.txt"


fact_dataset = TaskDataset(facts_train_path)  # max_n_facts=10)

noise_dataset = datasets.load_from_disk(noise_path_test)
noise_sampler = RetrSentenceSampler(noise_dataset)

dataset = RetrNoiseInjectionDataset(
    task_dataset=fact_dataset,
    noise_sentence_sampler=noise_sampler,
    num_sentences=num_sentences
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
from rl.babilong_env import BabilongEnv
# from rl.sacd import SAC, SACArgs
from rl.sarsa import SARSA, SARSAArgs
from rl.text_env import TextReplayBuffer
from transformers import AutoModel, AutoTokenizer
from rl.bert_predictor import BertPredictor
from rl.text_env import TextEnv, TextReplayBuffer


bert_name = "haisongzhang/roberta-tiny-cased"
tokenizer = AutoTokenizer.from_pretrained(bert_name, use_fast=True, revision="main")
bert_model = AutoModel.from_pretrained(bert_name, revision="main")


action_embed = BertPredictor(bert_model, 4, tokenizer, 512, 256, 1).cuda()
action_embed_target = BertPredictor(bert_model, 4, tokenizer, 512, 256, 1).cuda()

state_embed = BertPredictor(bert_model, 4, tokenizer, 512, 256, 1).cuda()
state_embed_target = BertPredictor(bert_model, 4, tokenizer, 512, 256, 1).cuda()

agent = SARSA(
    state_embed, action_embed, state_embed_target, action_embed_target, 
    SARSAArgs(gamma=0.95, tau=0.005,  lr=3e-4)
)


env = BabilongEnv( 
    embedder=agent.action_embed_target, 
    embed_tokenizer=tokenizer, 
    dataset=dataset,
    max_steps=max_steps
)

buffer = TextReplayBuffer(100_000, tokenizer = tokenizer)

In [6]:
from tqdm import tqdm
import torch

learning_start = 10_000

step = 0
R = 0
for ep_number in tqdm(range(100_000)):

    s = env.reset()
    done = False
    a_embeds = env.get_extra_embeds(agent.critic.action_embed)
    agent.policy.update(agent.critic)

    qf_loss, alpha_loss = 0, 0

    a_all = []

    while not done:
        step += 1
        
        action = agent.select_action(s, a_embeds, random = step < learning_start or step % 10 == 0)
        s_next, a_data, reward, done = env.step(action)
        buffer.add(s, a_data, s_next, reward, done, 0)
        s = s_next
        R += reward
        a_all.append(action)
        
        if step > learning_start and step % 5 == 0:
            s_batch, a_batch, next_s_batch, r_batch, not_done_batch, entropy_batch = buffer.sample(32)
            qf_loss = agent.update(s_batch, a_batch, next_s_batch, r_batch, not_done_batch, step)
    
    if ep_number % 200 == 0 and ep_number > 0:
        print(R / 200, qf_loss)
        print(a_all, env.ref_ids)
        R = 0

  0%|          | 0/100000 [00:00<?, ?it/s]

  0%|          | 204/100000 [00:06<1:01:09, 27.19it/s]

0.04 0
[15, 12, 4, 17, 7, 39, 21, 10, 11, 34] [47 49]


  0%|          | 405/100000 [00:13<48:46, 34.03it/s]  

0.03 0
[40, 42, 14, 47, 30, 20, 44, 33, 45, 13] [45 49]


...

0.21 0.013900350779294968
[49, 42, 47, 44, 41, 46, 33, 45] [45 49]


  5%|▌         | 5401/100000 [19:37<7:13:31,  3.64it/s]

0.23 0.021932901814579964
[28, 23, 24, 20, 7, 41, 47, 36, 30, 38] [39 48]


  5%|▌         | 5403/100000 [19:37<5:43:41,  4.59it/s]


KeyboardInterrupt: 